In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms
import numpy as np
import os, time

In [ ]:
# Generate random test inputs
num_samples = 100
X = torch.rand((num_samples), dtype=torch.float32)
R = torch.rand((num_samples), dtype=torch.float32)
ones = torch.ones((num_samples), dtype=torch.float32)
dataset = torch.stack([torch.log10(X), torch.log10(R), torch.log10(X)*torch.log10(R)], dim=1)

# Test CPU

In [ ]:
from espertaTorch import MultiEsperta
# Configuration matching original models
ESPERTA_CONFIGS = [
    # s1 models
    ([-6.07, -1.75, 1.14, 0.56], 0.28),  # w20_120
    ([-7.44, -2.99, 1.21, 0.69], 0.28),  # e40_19
    ([-5.02, -1.74, 0.64, 0.40], 0.23),  # e120_41
    
    # s2 models
    ([-6.07, -1.75, 1.14, 0.56], 0.35),  # w20_120
    ([-7.44, -2.99, 1.21, 0.69], 0.28),  # e40_19
    ([-5.02, -1.74, 0.64, 0.40], 0.23),  # e120_41
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultiEsperta(ESPERTA_CONFIGS)

In [ ]:
# Get outputs from PyTorch model
inference_time = time.time()
pytorch_outputs = model(dataset).detach().numpy()
inference_time = time.time() - inference_time

avg_inference_time = inference_time / num_samples
fps = num_samples / inference_time

print(f"The total inference time was {inference_time:.6f} seconds for {num_samples}.")
print("Average inference time per image: {:.6f} seconds".format(avg_inference_time))
print("FPS: {:.2f}".format(fps))

# Test DPU

In [ ]:
# VitisAI does not support the following operations:
# DPU does not support sigmoid
# aten::gt can't be converted to XIR.

# Test FINN

In [ ]:
# FINN does not support the following operations:

# Test HLS

In [ ]:
from pynq import Overlay
ol = Overlay("hls_m_axi.bit")
ESPERTA_ip = ol.entry_0

In [ ]:
print(ESPERTA_ip.register_map)
def load_input_to_ESPERTA(input_data):
    ESPERTA_ip.write(0x10, input_data[0])
    ESPERTA_ip.write(0x18, input_data[1])
    ESPERTA_ip.write(0x20, input_data[2])

def get_ESPERTA_output():
    return ESPERTA_ip.read(0x30), ESPERTA_ip.read(0x38), ESPERTA_ip.read(0x40)

def start_ESPERTA():
    ESPERTA_ip.write(0x00, 0x01)
    while ESPERTA_ip.read(0x00) & 0x4 == 0:
        continue

def run_ESPERTA(input_data):
    load_input_to_ESPERTA(input_data)
    start_ESPERTA()
    return get_ESPERTA_output()

In [ ]:
inference_time = 0
hls_outputs = []
for data in dataset:
    load_input_to_ESPERTA(data)
    start_time = time.time()
    start_ESPERTA()
    inference_time += time.time() - start_time
    out_data = get_ESPERTA_output()
    hls_outputs.append(out_data)

# Convert list to numpy array
hls_outputs = np.array(hls_outputs)

avg_infer_time = inference_time/num_samples
fps = num_samples/inference_time
print(f"The total inference time was {inference_time:.6f} seconds for {num_samples}.")
print("Average inference time: {} s".format(avg_infer_time))
print("Performance: {} FPS".format(fps))

In [ ]:
model_names = [
    "S1 W20_120",
    "S1 E40_19",
    "S1 E120_41",
    "S2 W20_120",
    "S2 E40_19",
    "S2 E120_41"
]

# Compare outputs using MSE and check if they're the same
mse = np.mean((pytorch_outputs - hls_outputs)**2, axis=0)
is_same = np.allclose(pytorch_outputs, hls_outputs, rtol=1e-5, atol=1e-5)

print("Comparison of PyTorch vs Original models:")
print("=" * 50)
print(f"Are all outputs the same? {'Yes' if is_same else 'No'}")
print("-" * 50)
for i, name in enumerate(model_names):
    print(f"{name}:")
    print(f"  Mean Square Error: {mse[i]:.8e}")
    is_model_same = np.allclose(pytorch_outputs[:, i], hls_outputs[:, i], rtol=1e-5, atol=1e-5)
    print(f"  Outputs match: {'Yes' if is_model_same else 'No'}")